# Citi Velocity USTs — intraday

Same MDP → TimeseriesBuilder → Query path as the EOD notebook, at 1-minute
resolution.

**The one thing to know about intraday here:** `CVTSHIST` silently downsamples by
requested *span*, not by age. The `MI01` cliff is measured at exactly **6 days** —
a 7-day request returns 10-minute rows that look identical to 1-minute rows. Every
fetch in this stack is held under that bound and its spacing verified, so you get
true minutes; if you fetch raw tags yourself, use
`MDP.CitiVelocityExcel.windowed.fetch_windowed` rather than one wide request.

The bond tape is **sparse**: 349 bonds warmed over 2026-08-05..07 gave 1,227
one-minute gaps out of 2,765, with the rest 2-4 minutes. That is liquidity, not
resolution — every stamp sits on a 1-minute boundary.

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

import sys
sys.path.append("../../")

import pandas as pd
from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue

usts_mdp = FixedRateBondsMDP(source="USTS_CITIVELO-RL")
ts_builder = TimeseriesBuilder()

## One session, one bond

Keep the window to a session or two. The FRB timeseries builder prices per
reference point, so a 1-minute range is one pricer build per minute — a couple of
hours is quick, a fortnight is not.

In [3]:
start = NYC_tz.localize(datetime.datetime(2026, 8, 7, 9, 30))
end   = NYC_tz.localize(datetime.datetime(2026, 8, 7, 12, 0))

q1 = UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_YTM)

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q1],
    routers={
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
    },
    n_jobs=12,
    freq="1min",
    ignore_cache_miss=True,
)
df.dropna()

FETCHING PRICERS:   0%|          | 0/151 [00:00<?, ?it/s]c:\Users\chris\anaconda3\envs\stir\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.fiscaldata.treasury.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
FETCHING PRICERS:  69%|██████▉   | 104/151 [08:03<03:38,  4.65s/it]
CitiVelocityExcelClient.drop_window_sheet: Excel refused the delete ((-2147023170, 'The remote procedure call failed.', None, None)). Leaving the sheet in place - the window's data is already read.
dropping window sheet failed: (-2147023174, 'The RPC server is unavailable.', None, None)
dropping window sheet failed: Excel is no longer reachable ((-2147023174, 'The RPC server is unavailable.', None, None)). Not retrying: retrying into a dying OLE server is what wedges it, and recovery costs a full add-in re-login.
dropping window shee

""
Date


In [4]:
plot, fig, _, _, legend = make_secondary_axis_plot(engine="plotly")
plot(df[q1.col_name()].dropna(), which="left")
legend(valfmt="{:.4f}", show_date=True)

KeyError: 'CT10 OUTRIGHT YTM'

## Reading the raw intraday tape

Straight off the warmed cache, with no pricing in between — useful when you want
Citi's published `PRICE`/`YIELD` rather than a locally re-solved yield.
`offline=True` means it cannot reach Excel at all, so anything served here is
genuinely cached.

In [ ]:
from MDP.CitiVelocityExcel.bonds.resolution import resolve_bond
from MDP.CitiVelocityExcel.quotes import CitiVeloQuotes
from MDP.CitiVelocityExcel import tags as T

r = resolve_bond("91282CNJ6")
quotes = CitiVeloQuotes(offline=True)

tape = quotes.frame(
    [T.bond(r.isin, "PRICE"), T.bond(r.isin, "YIELD")],
    "MI01",
    start=datetime.datetime(2026, 8, 6),
    end=datetime.datetime(2026, 8, 7, 23, 59),
)
print(f"{r.isin}  {r.descriptor.description}: {len(tape)} rows")

spacing = pd.Series(tape.index).diff().dropna()
print("min spacing:", spacing.min(), "| 1-min gaps:", (spacing == pd.Timedelta(minutes=1)).sum())
tape.tail()

## Snapshot at an instant

`snapshot_from_frame` resolves one instant with `asof` (backward-only) by
default. `nearest` is direction-unbounded and was measured answering a 00:05 ET
request with a print 55 minutes in the **future**, so it is opt-in.

In [ ]:
from MDP.CitiVelocityExcel.quotes import snapshot_from_frame

when = datetime.datetime(2026, 8, 7, 11, 15)
row = snapshot_from_frame(tape, when, method="asof")
print(f"as of {when}:")
row